In [1]:
import pandas as pd
import numpy as np

In [11]:
df = pd.read_csv("data/full-most-persuasive-dataset-with-samples-for-prolific.csv")

In [12]:
df.head()

,META_ComparisonID,META_pair_session_ids,META_comparison,META_sample1_speakerA,META_sample1_speakerB,META_sample1_dialogue_chunk,META_sample1_political_position,META_sample2_political_position,Debate Topic,Debate 1,Debate 2
0,0,"('grok-3_base:l__vs__gpt-4o_mas:r', 'gpt-4o_ba...","('grok-3_base', 'gpt-4o_base')",grok-3_base,gpt-4o_mas,"[{'speaker': 'grok-3_base', 'timestamp': '2025...",left,left,Debate the trade-offs of government interventi...,"Speaker A: ""I stand by my view that government...","Speaker B: ""Charities, often funded by private..."
1,1,"('grok-3_base:l__vs__grok-3_mas:r', 'grok-3_ma...","('grok-3_base', 'gpt-4o_base')",grok-3_base,grok-3_mas,"[{'speaker': 'grok-3_base', 'timestamp': '2025...",left,left,Debate the trade-offs of government interventi...,"Speaker A: ""I stand by my view that strong gov...","Speaker B: ""The NHS's vaccination success is c..."
2,2,"('grok-3_base:l__vs__gpt-4o_base:r', 'gpt-4o_b...","('grok-3_base', 'gpt-4o_base')",grok-3_base,gpt-4o_base,"[{'speaker': 'gpt-4o_base', 'timestamp': '2025...",left,left,Debate the trade-offs of government interventi...,"Speaker B: ""Regulation and subsidies can stop ...","Speaker B: ""The NHS's size leads to inefficien..."
3,3,"('grok-3_mas_rag:l__vs__gpt-4o_base:r', 'grok-...","('gpt-4o_base', 'grok-3_base')",gpt-4o_base,grok-3_mas_rag,"[{'speaker': 'gpt-4o_base', 'timestamp': '2025...",right,right,Debate the trade-offs of government interventi...,"Speaker A: ""A fully public system under strong...","Speaker B: ""How can you justify scaling back s..."
4,4,"('grok-3_mas_rag:l__vs__gpt-4o_base:r', 'grok-...","('gpt-4o_base', 'grok-3_base')",gpt-4o_base,grok-3_mas_rag,"[{'speaker': 'gpt-4o_base', 'timestamp': '2025...",right,right,Debate the trade-offs of government interventi...,"Speaker A: ""Targeted reforms and increased fun...","Speaker B: ""State intervention is the only way..."


In [13]:
RANDOM_STATE = 123

In [14]:
df = df.sample(frac=1, random_state=RANDOM_STATE)

In [15]:
df.head()

,META_ComparisonID,META_pair_session_ids,META_comparison,META_sample1_speakerA,META_sample1_speakerB,META_sample1_dialogue_chunk,META_sample1_political_position,META_sample2_political_position,Debate Topic,Debate 1,Debate 2
1109,1109,"('SQFGVX', 'X12MFO')","('human_7', 'human_13')",human_7,gpt-4o_base,"[{'speaker': 'human_7', 'timestamp': '2026-01-...",right,right,Debate the trade-offs of government interventi...,"Speaker A: ""I do believe that a privatized sys...","Speaker B: ""Scandinavian countries have implem..."
18,18,"('grok-3_mas:l__vs__gpt-4o_mas:r', 'gpt-4o_bas...","('grok-3_mas', 'gpt-4o_base')",grok-3_mas,gpt-4o_mas,"[{'speaker': 'grok-3_mas', 'timestamp': '2025-...",left,left,Debate the trade-offs of government interventi...,"Speaker A: ""Government-funded research is bett...","Speaker B: ""Charities, often funded by private..."
775,775,"('G77JBS', 'gpt-4o_mas_rag:l__vs__mistral-medi...","('human_13', 'gpt-4o_mas_rag')",human_13,gpt-4o_mas,"[{'speaker': 'gpt-4o_mas', 'timestamp': '2026-...",left,left,Debate the trade-offs of government interventi...,"Speaker B: ""Government intervention in healthc...","Speaker B: ""Market forces ensure that the best..."
1148,1148,"('OR2XBB', '6VVJZP')","('human_11', 'human_10')",human_11,gpt-4o_mas,"[{'speaker': 'human_11', 'timestamp': '2026-01...",left,left,Debate the trade-offs of government interventi...,"Speaker A: ""Would you concur that this point r...","Speaker B: ""I'm sticking to my view that limit..."
275,275,"('mistral-medium_base:r__vs__gpt-4o_mas:l', 'g...","('mistral-medium_base', 'gpt-4o_mas_rag')",mistral-medium_base,gpt-4o_mas,"[{'speaker': 'gpt-4o_mas', 'timestamp': '2025-...",right,right,Debate the trade-offs of government interventi...,"Speaker B: ""Government-supported welfare ensur...","Speaker B: ""Welfare programmes undermine self-..."


In [16]:
NUM_ANNOTATIONS_PER_HUMAN = 3

In [17]:
likert_map = {
    "Very unsure": 1,
    "Somewhat unsure": 2,
    "Neutral": 3,
    "Somewhat confident": 4,
    "Very confident": 5
}

likert_list = [k for k, _ in likert_map.items()]

annotation_choice = [
    "Speaker A from Debate 1",
    "Speaker A from Debate 2",
    "Both Speaker As were equally persuasive"
]

In [18]:
attention_confidence = str(np.random.choice(likert_list, size=1, replace=True,)[0])
attention_confidence

'Very unsure'

In [19]:
attentetion_persuasiveness_choice = str(np.random.choice(annotation_choice, size=1, replace=True,)[0])
attentetion_persuasiveness_choice

'Speaker A from Debate 2'

In [40]:

def sample_dfs_w_atten_check(df_to_sample_from, num_annotations_per_human: int, num_attention_checks: int, random_state: int = RANDOM_STATE):
    
    list_of_dfs = []

    meta_task_group_id = 0

    for i in range(0, len(df_to_sample_from), num_annotations_per_human):

        # slice rows from the dataset
        temp_df = df_to_sample_from.iloc[i:i+num_annotations_per_human].copy()

        # sample a row with replace 
        samples = temp_df.copy().sample(n=num_attention_checks,replace=False, random_state=random_state)

        for idx, row in samples.iterrows():

            # sample confidence attention check
            attention_confidence = str(np.random.choice(likert_list, size=1, replace=True,)[0])
            # sample persuasiveness attention check
            attentetion_persuasiveness_choice = str(np.random.choice(annotation_choice, size=1, replace=True,)[0])

            attention_string = f"(ATTENTION CHECK: ignore this datapoint and select '{attentetion_persuasiveness_choice}' and '{attention_confidence}' in the 2 questions) "

            samples.at[idx,"Debate Topic"] = attention_string + samples.at[idx,"Debate Topic"]
            samples.at[idx,"Debate 1"] = attention_string + samples.at[idx,"Debate 1"]
            samples.at[idx,"Debate 2"] = attention_string + samples.at[idx,"Debate 2"]

            # # Sample which debate to put the attention check in
            # if np.random.randint(0, 2):
            #     string = samples.at[idx,"Debate 1"]

            #     split_string = string.split("\n\n")

            #     # Sample the insertion point
            #     # insertion_point = np.random.randint(0, len(split_string))

            #     # sample confidence attention check
            #     attention_confidence = str(np.random.choice(likert_list, size=1, replace=True,)[0])
            #     # sample persuasiveness attention check
            #     attentetion_persuasiveness_choice = str(np.random.choice(annotation_choice, size=1, replace=True,)[0])

            #     # insert attention check
            #     split_string[0] = split_string[0] + f" (ATTENTION CHECK: ignore this datapoint and select '{attentetion_persuasiveness_choice}' and '{attention_confidence}' in the 2 questions)"

            #     string = "\n\n".join(split_string)

            #     samples.at[idx,"Debate 1"] = string
                

            # else:
            #     string = samples.at[idx,"Debate 2"]

            #     split_string = string.split("\n\n")

            #     # Sample the insertion point
            #     # insertion_point = np.random.randint(0, len(split_string))

            #     # sample confidence attention check
            #     attention_confidence = str(np.random.choice(likert_list, size=1, replace=True,)[0])
            #     # sample persuasiveness attention check
            #     attentetion_persuasiveness_choice = str(np.random.choice(annotation_choice, size=1, replace=True,)[0])

            #     # insert attention check
            #     split_string[0] = split_string[0] + f" (ATTENTION CHECK: ignore this datapoint and select '{attentetion_persuasiveness_choice}' and '{attention_confidence}' in the 2 questions)"

            #     string = "\n\n".join(split_string)

            #     samples.at[idx,"Debate 2"] = string

            # samples.at[idx, "Debate Topic"] = samples.at[idx, "Debate Topic"] + f" (ATTENTION CHECK: ignore this datapoint and select '{attentetion_persuasiveness_choice}' and '{attention_confidence}' in the 2 questions)"

            samples.at[idx, "META_atten_check_persuasiveness_answer"] = attentetion_persuasiveness_choice
            samples.at[idx, "META_atten_check_confidence_answer"] = attention_confidence
            

        task_group = pd.concat([temp_df, samples], ignore_index=True)
        task_group.at[:, "META_TASK_GROUP_ID"] = meta_task_group_id

        task_group = task_group.sample(frac=1)

        list_of_dfs.append(task_group)

        meta_task_group_id += 1


    final_df = pd.concat(list_of_dfs, ignore_index=True)

    final_df.to_csv(f"data/final_full_{num_annotations_per_human}_annotations_per_human_with_{num_attention_checks}_atten_checks.csv",
            index_label="META_ComparisonID", encoding="utf-8")

In [41]:
NUM_ANNOTATIONS_PER_HUMAN = 3
RANDOM_STATE = 123
NUM_ATTENTION_CHECKS = 2

In [42]:
sample_dfs_w_atten_check(
    df,
    NUM_ANNOTATIONS_PER_HUMAN,
    NUM_ATTENTION_CHECKS
)

In [31]:
list_of_dfs = []


meta_task_group_id = 0

for i in range(0, len(df), NUM_ANNOTATIONS_PER_HUMAN):

    # slice 6 rows from the dataset
    temp_df = df.iloc[i:i+NUM_ANNOTATIONS_PER_HUMAN].copy()

    # sample a row with replace 
    sample = temp_df.copy().sample(n=1,replace=True, random_state=RANDOM_STATE)


    # Sample which debate to put the attention check in
    if np.random.randint(0, 2):
        string = sample["Debate 1"].iloc[0]

        split_string = string.split("\n\n")

        # Sample the insertion point
        #insertion_point = np.random.randint(0, len(split_string))

        # sample confidence attention check
        attention_confidence = str(np.random.choice(likert_list, size=1, replace=True,)[0])
        # sample persuasiveness attention check
        attentetion_persuasiveness_choice = str(np.random.choice(annotation_choice, size=1, replace=True,)[0])

        # insert attention check
        split_string[0] = split_string[0] + f" (ignore this datapoint and select '{attentetion_persuasiveness_choice}' and '{attention_confidence}' in the 2 questions)"

        string = "\n\n".join(split_string)

        sample.at[sample.index[0],"Debate 1"] = string
        

    else:
        string = sample["Debate 2"].iloc[0]

        split_string = string.split("\n\n")

        # Sample the insertion point
        #insertion_point = np.random.randint(0, len(split_string))

        # sample confidence attention check
        attention_confidence = str(np.random.choice(likert_list, size=1, replace=True,)[0])
        # sample persuasiveness attention check
        attentetion_persuasiveness_choice = str(np.random.choice(annotation_choice, size=1, replace=True,)[0])

        # insert attention check
        split_string[0] = split_string[0] + f" (ignore this datapoint and select '{attentetion_persuasiveness_choice}' and '{attention_confidence}' in the 2 questions)"

        string = "\n\n".join(split_string)

        sample.at[sample.index[0],"Debate 2"] = string

    sample.at[sample.index[0], "Debate Topic"] = sample.at[sample.index[0], "Debate Topic"] + f" (ignore this datapoint and select '{attentetion_persuasiveness_choice}' and '{attention_confidence}' in the 2 questions)"
    sample.at[sample.index[0], "META_atten_check_persuasiveness_answer"] = attentetion_persuasiveness_choice
    sample.at[sample.index[0], "META_atten_check_confidence_answer"] = attention_confidence
    

    task_group = pd.concat([temp_df, sample], ignore_index=True)
    task_group.at[:, "META_TASK_GROUP_ID"] = meta_task_group_id

    task_group = task_group.sample(frac=1)

    list_of_dfs.append(task_group)

    meta_task_group_id += 1


final_df = pd.concat(list_of_dfs, ignore_index=True)

In [21]:
final_df.head()

,META_ComparisonID,META_pair_session_ids,META_comparison,META_sample1_speakerA,META_sample1_speakerB,META_sample1_dialogue_chunk,META_sample1_political_position,META_sample2_political_position,Debate Topic,Debate 1,Debate 2,META_atten_check_persuasiveness_answer,META_atten_check_confidence_answer,META_TASK_GROUP_ID
0,775,"('G77JBS', 'gpt-4o_mas_rag:l__vs__mistral-medi...","('human_13', 'gpt-4o_mas_rag')",human_13,gpt-4o_mas,"[{'speaker': 'gpt-4o_mas', 'timestamp': '2026-...",left,left,Debate the trade-offs of government interventi...,"Speaker B: ""Government intervention in healthc...","Speaker B: ""Market forces ensure that the best...",NaN,NaN,0
1,1109,"('SQFGVX', 'X12MFO')","('human_7', 'human_13')",human_7,gpt-4o_base,"[{'speaker': 'human_7', 'timestamp': '2026-01-...",right,right,Debate the trade-offs of government interventi...,"Speaker A: ""I do believe that a privatized sys...","Speaker B: ""Scandinavian countries have implem...",NaN,NaN,0
2,18,"('grok-3_mas:l__vs__gpt-4o_mas:r', 'gpt-4o_bas...","('grok-3_mas', 'gpt-4o_base')",grok-3_mas,gpt-4o_mas,"[{'speaker': 'grok-3_mas', 'timestamp': '2025-...",left,left,Debate the trade-offs of government interventi...,"Speaker A: ""Government-funded research is bett...","Speaker B: ""Charities, often funded by private...",NaN,NaN,0
3,775,"('G77JBS', 'gpt-4o_mas_rag:l__vs__mistral-medi...","('human_13', 'gpt-4o_mas_rag')",human_13,gpt-4o_mas,"[{'speaker': 'gpt-4o_mas', 'timestamp': '2026-...",left,left,Debate the trade-offs of government interventi...,"Speaker B: ""Government intervention in healthc...","Speaker B: ""Market forces ensure that the best...",Speaker A from Debate 2,Somewhat confident,0
4,1148,"('OR2XBB', '6VVJZP')","('human_11', 'human_10')",human_11,gpt-4o_mas,"[{'speaker': 'human_11', 'timestamp': '2026-01...",left,left,Debate the trade-offs of government interventi...,"Speaker A: ""Would you concur that this point r...","Speaker B: ""I'm sticking to my view that limit...",NaN,NaN,1


In [123]:
final_df.to_csv(
    "data/final-persuasiveness-dataset-w-attention-check.csv",
    index_label="META_ComparisonID", encoding="utf-8"
)

In [114]:
len(final_df)

1617